In [15]:
import jax, jax.numpy as jnp

In [16]:
@jax.jit
def attention(h, s): #RNN style
    #h(0, ..., m) - m->internal state, [n, len]
    #s(0, ..., n) - n->output, [m, len]
    #h_m ~ s_0
    e = s @ h.T # [n, m]
    alpha = jax.nn.softmax(e, axis=1)
    alpha = alpha @ h
    return jnp.concatenate((s, alpha), axis=1)

In [17]:
@jax.jit
def attention(h, s, W_a, U_a, v_a): #Canonical style
    #h(0, ..., m-1) - encoder states (bi-RNN, forward+backward concat), [m, dh]
    #s(0, ..., n-1) - decoder states s_{i-1} (state BEFORE the step), [n, ds]
    #W_a - projection of s into alignment space, [ds, da]
    #U_a - projection of h into alignment space, [dh, da]
    #v_a - scoring vector, turns a da-vector into one number, [da]

    s_p = s @ W_a                                   #[n, da]
    h_p = h @ U_a                                   #[m, da]
    e = jnp.tanh(s_p[:, None, :] + h_p[None, :, :]) @ v_a   #[n, m]; e[i, j] = how much h_j is needed at step i
    alpha = jax.nn.softmax(e, axis=1)               #[n, m]; weights over encoder positions, each row sums to 1
    c = alpha @ h                                   #[n, dh]; context c_i = sum_j alpha_ij * h_j
    return c, alpha                                 #c goes into the decoder RNN cell: s_i = f(s_{i-1}, y_{i-1}, c_i)

In [14]:
@jax.jit
def self_attention(x, wq, wk, wv):
    # x[len, emb_size]
    # key = jax.random.key(0)
    # kq, kk, kv = jax.random.split(key, 3)
    # wq, wk, wv = jax.random.uniform(kq, shape=(x.shape[1], 1024)), jax.random.uniform(kk, shape=(x.shape[1], 1024), jax.random.uniform(kv, shape=(x.shape[1], 1024)
    Q, K, V = x @ wq, x @ wk, x @ wv

    i = jnp.arange(x.shape[0])[:, None] #[len, 1] row = query
    j = jnp.arange(x.shape[0])[None, :] #[1, len] col = key
    causal_mask = jnp.where(i >= j, jnp.zeros((x.shape[0], x.shape[0])), -jnp.inf)

    attention = jax.nn.softmax( (Q @ K.T + causal_mask) / jnp.sqrt(Q.shape[1]), axis=1 ) @ V

    return attention #[len, 1024] 

Array([[14, 14],
       [14, 14]], dtype=int32)

In [25]:
jnp.arange(64)

Array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63], dtype=int32)